# 03_candidate_generation — 候補ソース別のRecall@12監査

Phase 3の狙い: ランキングモデル(Phase 4)は「候補集合の中を並べ替える」だけなので、
候補のRecall(実購入商品をどれだけ候補に含められているか)が低いと後工程で頭打ちになる。
そこで複数の候補ソースを個別に作り、共通のバックテスト基盤(`common.py`)でRecall@12を単体評価してから、
Phase 4(特徴量+LightGBM)に進む。

候補生成ロジックは `candidates.py` に共通化してあり、04以降のノートブックもこれを再利用する。

## 候補ソース

- **A. 自己リピート**: 過去に購入したarticle_idを直近購入日→頻度順(H1)
- **B. トレンド人気**: cutoff直前1週間の全体人気トップ12(H10の死に筋フィルタも兼ねる)
- **C. 同カテゴリ/同色**: 過去に購入したproduct_codeの色違い(H1の9.1%)を最優先し、不足分は顧客の最頻section_name内の人気商品で埋める(H2c)
- **D. 協調フィルタリング**: cutoff以前のuser-item購入回数行列にimplicit ALSを適用したトップ12
- **E. アイテム共起**: 同一顧客・同一日(バスケット、H8)の同時購入からitem-to-item共起を作り、顧客の自己リピート上位12件をシードにして「一緒によく買われる商品」を集計したトップ12
- **F. テキストコンテンツ類似**: `prod_name`/`product_type_name`/`colour_group_name`/`detail_desc`のTF-IDFコサイン類似度(`build_content_similarity.py`で事前計算、カタログのみに依存しcutoff非依存)で、自己リピート上位12件をシードに内容が似た商品を集計したトップ12(新規追加)

いずれもcutoff以前のデータのみで計算し、02_baseline.ipynbと同じ「直近7日間ホールドアウト」バックテストで評価する(Fの類似度テーブル自体はカタログ全体で事前計算済みだが、シードはcutoff以前の購入履歴から作るため、実質的にリークはない)。

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("."))
from common import backtest_split, get_con, mapk, mean_recall_at_k
from candidates import (
    build_category_color_candidates,
    build_cf_candidates,
    build_content_similarity_candidates,
    build_cooccurrence_candidates,
    build_popularity_candidates,
    build_repeat_candidates,
)

con = get_con()
cutoff, actual_map = backtest_split(con, holdout_days=7)
print(f"cutoff = {cutoff}")
print(f"バックテスト期間に購入があった顧客数: {len(actual_map):,}")

## ソースA: 自己リピート(H1)

In [ ]:
repeat_map = build_repeat_candidates(con, cutoff, target_customers=actual_map.keys(), k=12)
pred_map_repeat = {cid: list(repeat_map.get(cid, [])) for cid in actual_map}
print(f"自己リピート候補を持つ顧客数(バックテスト対象中): {sum(1 for v in pred_map_repeat.values() if len(v) > 0):,} / {len(actual_map):,}")

## ソースB: トレンド人気(H10)

In [ ]:
pop_list = build_popularity_candidates(con, cutoff, top_n=12)
pred_map_pop = {cid: pop_list for cid in actual_map}
print(f"トレンド人気トップ12: {pop_list}")

## ソースC: 同カテゴリ/同色(H1色違い + H2cセグメント人気)

In [ ]:
category_map = build_category_color_candidates(con, cutoff, target_customers=actual_map.keys(), k=12)
pred_map_category = {cid: list(category_map.get(cid, [])) for cid in actual_map}
print(f"同カテゴリ/同色候補を持つ顧客数(バックテスト対象中): {sum(1 for v in pred_map_category.values() if len(v) > 0):,} / {len(actual_map):,}")

## ソースD: 協調フィルタリング(implicit ALS)

cutoff以前のuser-item購入回数を信頼度(confidence)としたimplicit feedback ALS。
他ソースと公平に比較するため、`filter_already_liked_items=False`にして既購入商品を除外しない。
バックテスト対象顧客(直近1週間に購入があった約7万人)分だけレコメンドを計算する。

In [ ]:
cf_map = build_cf_candidates(con, cutoff, actual_map.keys(), k=12)
pred_map_cf = {cid: list(cf_map.get(cid, [])) for cid in actual_map}
print(f"CF候補を作成した顧客数: {sum(1 for v in pred_map_cf.values() if len(v) > 0):,} / {len(actual_map):,}")

## ソースE: アイテム共起(H8)

顧客の自己リピート上位12件(ソースAと同じシード)をもとに、
同一バスケット(同一顧客・同一日)でよく一緒に買われる商品を集計する。
自己リピート候補が無い顧客(コールドスタート)はこのソースも空になる。

In [ ]:
cooc_map = build_cooccurrence_candidates(con, cutoff, actual_map.keys(), k=12, seed_map=repeat_map)
pred_map_cooc = {cid: list(cooc_map.get(cid, [])) for cid in actual_map}
print(f"共起候補を持つ顧客数(バックテスト対象中): {sum(1 for v in pred_map_cooc.values() if len(v) > 0):,} / {len(actual_map):,}")

## ソースF: テキストコンテンツ類似

顧客の自己リピート上位12件(ソースAと同じシード)をもとに、`build_content_similarity.py`で事前計算した
TF-IDFコサイン類似度が高い商品を集計する。自己リピート候補が無い顧客(コールドスタート)はこのソースも空になる。

In [ ]:
content_map = build_content_similarity_candidates(con, cutoff, actual_map.keys(), k=12, seed_map=repeat_map)
pred_map_content = {cid: list(content_map.get(cid, [])) for cid in actual_map}
print(f"コンテンツ類似候補を持つ顧客数(バックテスト対象中): {sum(1 for v in pred_map_content.values() if len(v) > 0):,} / {len(actual_map):,}")

## 比較: 各ソース単体のRecall@12 / MAP@12

- Recall@12: 実際の購入商品のうち、そのソースの上位12件に含まれていた割合の顧客平均
- 参考として、02_baseline.ipynb と同じ「自己リピート→トレンド人気で穴埋め」ハイブリッドと、
  6ソースを単純結合した候補プール(最大72件、まだ12件に絞る前)のRecallも載せる。
  後者は「Phase 4のランキングモデルが理論上狙える上限」の目安になる。

In [ ]:
results = []
for name, pred_map in [
    ("A: 自己リピート", pred_map_repeat),
    ("B: トレンド人気", pred_map_pop),
    ("C: 同カテゴリ/同色", pred_map_category),
    ("D: 協調フィルタリング(ALS)", pred_map_cf),
    ("E: アイテム共起", pred_map_cooc),
    ("F: テキストコンテンツ類似", pred_map_content),
]:
    results.append({
        "source": name,
        "coverage(候補あり顧客率)": np.mean([len(pred_map.get(cid, [])) > 0 for cid in actual_map]),
        "recall@12": mean_recall_at_k(actual_map, pred_map, k=12),
        "map@12": mapk(actual_map, pred_map, k=12),
    })

# 参考1: 02_baseline.ipynbと同じハイブリッド(自己リピート→トレンド人気で穴埋め)
pred_map_hybrid = {}
for cid in actual_map:
    candidates = list(pred_map_repeat.get(cid, []))
    if len(candidates) < 12:
        seen = set(candidates)
        for a in pop_list:
            if a not in seen:
                candidates.append(a)
                seen.add(a)
            if len(candidates) >= 12:
                break
    pred_map_hybrid[cid] = candidates[:12]

results.append({
    "source": "参考: 02_baselineハイブリッド(A→B)",
    "coverage(候補あり顧客率)": np.mean([len(pred_map_hybrid.get(cid, [])) > 0 for cid in actual_map]),
    "recall@12": mean_recall_at_k(actual_map, pred_map_hybrid, k=12),
    "map@12": mapk(actual_map, pred_map_hybrid, k=12),
})

# 参考2: 6ソースの候補プールを単純結合(重複除去、最大72件、12件には絞らない)
pred_map_union = {}
for cid in actual_map:
    seen = []
    seen_set = set()
    for pred_map in (pred_map_repeat, pred_map_pop, pred_map_category, pred_map_cf, pred_map_cooc, pred_map_content):
        for a in pred_map.get(cid, []):
            if a not in seen_set:
                seen.append(a)
                seen_set.add(a)
    pred_map_union[cid] = seen

results.append({
    "source": "参考: 6ソース単純結合プール(上限72件)",
    "coverage(候補あり顧客率)": np.mean([len(pred_map_union.get(cid, [])) > 0 for cid in actual_map]),
    "recall@12": mean_recall_at_k(actual_map, pred_map_union, k=72),
    "map@12": np.nan,  # 72件はMAP@12の定義と噛み合わないため参考値なし
})

results_df = pd.DataFrame(results)
results_df

## 次のステップ

- Recall@12が低いソースは単体では弱いが、結合プールのRecallが個別より大きく伸びていれば「候補源として補完的」であることを意味する → Phase 4でその特徴量/候補として活かす価値がある
- 結合プールのRecallがPhase 4で目指せる上限の目安
- 次は`04_feature_engineering.ipynb`で、5ソースをベースにuser/item/user-item特徴量を作り(+学習週を増やして季節性への頑健性を上げる)、`05_train_lgbm.ipynb`でLightGBM Ranker/Classifierを再学習する